## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.stats import spearmanr
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# Paths
input_dir = Path("input")
output_dir = Path("output")
input_ref = input_dir / "glambie_reference"
glambie_runs = input_dir / "glambie_runs_clean"
output_sensitivity = output_dir / "sensitivity"

output_sensitivity.mkdir(parents=True, exist_ok=True)

In [ ]:
# Colors
colors_list = list(plt.colormaps["tab10"].colors)
blue, orange = colors_list[:2]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=colors_list)

## Configuration

In [ ]:
# Region mapping: directory name -> display name
regions = {
    "1_alaska": "Alaska",
    "19_antarctic_and_subantarctic": "Antarctic and subantarctic islands",
    "3_arctic_canada_north": "Arctic Canada north",
    "4_arctic_canada_south": "Arctic Canada south",
    "5_greenland_periphery": "Greenland periphery",
    "6_iceland": "Iceland",
    "7_svalbard": "Svalbard and Jan Mayen",
    "8_scandinavia": "Scandinavia",
    "9_russian_arctic": "Russian Arctic",
    "10_north_asia": "North Asia",
    "11_central_europe": "Central Europe",
    "12_caucasus_middle_east": "Caucasus and Middle East",
    "13_central_asia": "Central Asia",
    "14_south_asia_west": "South Asia west",
    "15_south_asia_east": "South Asia east",
    "16_low_latitudes": "Low latitudes",
    "17_southern_andes": "Southern Andes",
    "18_new_zealand": "New Zealand",
    "2_western_canada_us": "Western Canada and USA",
}

In [ ]:
# Load initial glacier mass data
glacier_mass_file = input_ref / "glacier_mass_2000.csv"
glacier_mass_df = pd.read_csv(str(glacier_mass_file), sep=";")
glacier_mass_dict = dict(zip(glacier_mass_df["Region"], glacier_mass_df["Mass"]))

print(f"Loaded initial mass for {len(glacier_mass_dict)} regions")
glacier_mass_df.head()

## Core functions

In [ ]:
def compute_all_metrics(df1, df2, glacier_mass_dict, region):
    """
    Compute comparison metrics between two runs.
    """
    # Get initial mass
    initial_mass = glacier_mass_dict.get(region)
    
    if initial_mass is None:
        raise ValueError(f"No initial mass found for region: {region}")
    
    # Merge timeseries
    merged = pd.merge(df1, df2, on="start_dates", suffixes=("_run1", "_run2"), how="outer")
    cumul_run1 = merged["combined_gt_run1"].sum()
    cumul_run2 = merged["combined_gt_run2"].sum()
    
    # Calculate final masses
    final_mass_run1 = initial_mass + cumul_run1
    final_mass_run2 = initial_mass + cumul_run2
    
    # Relative difference (% of initial mass)
    rel_diff = (final_mass_run2 - final_mass_run1) / initial_mass * 100
    
    # Absolute difference (Gt)
    abs_diff = cumul_run2 - cumul_run1
    
    # Spearman correlation
    try:
        corr, _ = spearmanr(merged["combined_gt_run1"], merged["combined_gt_run2"])
        if np.isnan(corr):
            corr = 0.0
    except Exception:
        corr = 0.0
    
    return rel_diff, abs_diff, corr

In [ ]:
def compare_runs(run1_path, run2_path, glacier_mass_dict, regions):
    """Compare two runs and return metrics for all regions."""
    region_names, rel_diffs, abs_diffs, corrs = [], [], [], []
    
    for region_dir, display_name in regions.items():
        csv1 = run1_path / "calendar_years" / f"{region_dir}.csv"
        csv2 = run2_path / "calendar_years" / f"{region_dir}.csv"
        
        if not csv1.exists() or not csv2.exists():
            continue
        
        df1, df2 = pd.read_csv(csv1), pd.read_csv(csv2)
        rel_diff, abs_diff, corr = compute_all_metrics(df1, df2, glacier_mass_dict, display_name)
        
        region_names.append(display_name)
        rel_diffs.append(rel_diff)
        abs_diffs.append(abs_diff)
        corrs.append(corr)
    
    # Global comparison
    csv1_g = run1_path / "calendar_years" / "0_global.csv"
    csv2_g = run2_path / "calendar_years" / "0_global.csv"
    if csv1_g.exists() and csv2_g.exists():
        df1, df2 = pd.read_csv(csv1_g), pd.read_csv(csv2_g)
        rel_diff, abs_diff, corr = compute_all_metrics(df1, df2, glacier_mass_dict, "Global")
        region_names.append("Global")
        rel_diffs.append(rel_diff)
        abs_diffs.append(abs_diff)
        corrs.append(corr)
    
    return region_names, rel_diffs, abs_diffs, corrs


def create_comparison_barchart(region_names, rel_diff_values, abs_diff_values, 
                               corr_values, run1_name, run2_name, output_prefix):
    """Create 3-panel horizontal barchart comparing runs."""
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 8), sharey=True)
    y_pos = range(len(region_names))
    
    # Panel 1: Relative difference
    ax1.barh(y_pos, rel_diff_values)
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels(region_names)
    ax1.invert_yaxis()
    ax1.set_xlabel("Relative Change (%)", fontsize=11)
    ax1.axvline(0, color="black", linewidth=0.8)
    
    max_abs_rel = max(abs(v) for v in rel_diff_values) or 1
    for i, val in enumerate(rel_diff_values):
        x_pos = val + (0.02 * max_abs_rel if val >= 0 else -0.02 * max_abs_rel)
        ax1.text(x_pos, i, f"{val:+.1f}%", va="center", ha="left" if val >= 0 else "right", fontsize=9)
    ax1.set_xlim(min(rel_diff_values) - 0.35*max_abs_rel, max(rel_diff_values) + 0.35*max_abs_rel)
    
    # Panel 2: Absolute difference
    ax2.barh(y_pos, abs_diff_values)
    ax2.set_xlabel("Absolute Difference (Gt)", fontsize=11)
    ax2.axvline(0, color="black", linewidth=0.8)
    
    max_abs_diff = max(abs(v) for v in abs_diff_values) or 1
    for i, val in enumerate(abs_diff_values):
        x_pos = val + (0.02 * max_abs_diff if val >= 0 else -0.02 * max_abs_diff)
        ax2.text(x_pos, i, f"{val:+.2f}", va="center", ha="left" if val >= 0 else "right", fontsize=9)
    ax2.set_xlim(min(abs_diff_values) - 0.30*max_abs_diff, max(abs_diff_values) + 0.30*max_abs_diff)
    
    # Panel 3: Correlation
    ax3.barh(y_pos, corr_values)
    ax3.set_xlabel("Spearman Correlation", fontsize=11)
    ax3.set_xlim(-0.1, 1.1)
    
    for i, val in enumerate(corr_values):
        ax3.text(val + 0.02, i, f"{val:.2f}", va="center", fontsize=9)
    
    fig.suptitle(f"{run1_name} vs {run2_name}", fontsize=13)
    fig.tight_layout()
    fig.savefig(output_sensitivity / f"{output_prefix}_barchart.png", dpi=200, bbox_inches="tight")
    print(f"Saved: {output_prefix}_barchart.png")
    plt.show()

## Compare two specific runs

In [ ]:
# Define runs to compare
run1_path = glambie_runs / "datasets_default"
run2_path = glambie_runs / "datasets_including_gravimetry"

print(f"Comparing: {run1_path.name} vs {run2_path.name}")

In [ ]:
# Compute metrics for all regions using the compare_runs function
region_names, rel_diff_values, abs_diff_values, corr_values = compare_runs(
    run1_path, run2_path, glacier_mass_dict, regions
)

print(f"Computed metrics for {len(region_names)} regions")

In [ ]:
# Create results table
results_df = pd.DataFrame({
    "Region": region_names,
    "Relative Difference (%)": rel_diff_values,
    "Absolute Difference (Gt)": abs_diff_values,
    "Spearman Correlation": corr_values
})

# Save table
table_path = output_sensitivity / "default_vs_gravimetry_table.csv"
results_df.to_csv(table_path, index=False)
print(f"Saved table to: {table_path}")

results_df

In [ ]:
# Create barchart
create_comparison_barchart(
    region_names,
    rel_diff_values,
    abs_diff_values,
    corr_values,
    run1_path.name,
    run2_path.name,
    "default_vs_gravimetry"
)

## Compare all runs against reference

In [ ]:
# Discover all runs
all_runs = sorted([
    p for p in glambie_runs.iterdir()
    if p.is_dir() and (p / "calendar_years").is_dir()
])

# Find reference run
reference_run = next((r for r in all_runs if "default" in r.name), all_runs[0])

print(f"Reference run: {reference_run.name}")
print(f"Found {len(all_runs)} total runs")

In [ ]:
# Compare all runs
all_results = {}

for run in all_runs:
    if run == reference_run:
        continue
    
    # Use compare_runs function
    region_names, rel_diff_values, abs_diff_values, corr_values = compare_runs(
        reference_run, run, glacier_mass_dict, regions
    )
    
    if not region_names:
        continue
    
    # Store results
    all_results[run.name] = {
        "region_names": region_names,
        "rel_diff_values": rel_diff_values,
        "abs_diff_values": abs_diff_values,
        "corr_values": corr_values
    }
    
    # Create individual barchart
    run_label = run.name.split("_", 1)[1]
    safe_name = run.name.replace(" ", "_").replace("/", "-")
    create_comparison_barchart(
        region_names,
        rel_diff_values,
        abs_diff_values,
        corr_values,
        reference_run.name,
        run_label,
        f"sensitivity_{safe_name}"
    )

print(f"\nProcessed {len(all_results)} runs")

In [ ]:
# Create summary table
summary_rows = []
for run_name, results in all_results.items():
    run_label = run_name.split("_", 1)[1]
    global_idx = results["region_names"].index("Global")
    
    summary_rows.append({
        "Run": run_label,
        "Relative Difference (%)": results["rel_diff_values"][global_idx],
        "Absolute Difference (Gt)": results["abs_diff_values"][global_idx],
        "Spearman Correlation": results["corr_values"][global_idx]
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(output_sensitivity / "run_influence_summary.csv", index=False)
print(f"Saved summary table to: {output_sensitivity / 'run_influence_summary.csv'}")

summary_df